# 🩸 Phase 3c — WBC-Only Triple Fusion

**A focused ablation of Phase 3 using ONLY White Blood Cell Count (WBC) as clinical metadata.**

## Hypothesis
WBC is the primary acute-phase biomarker of bacterial infection. As the top-ranked feature in Phase 3 permutation importance, **WBC alone may capture most of the clinical metadata signal** while avoiding feature noise from 15 less-informative variables (gender, alk_phos, creatinine, etc.).

## Architecture
```
[Chest X-ray]  →  DenseNet-121+CBAM  →  1024-d Image Embedding ──┐
                                                                    ├─→ 8-Head Cross-Attention ─┐
[Report Text]  →  Bio_ClinicalBERT   →  768-d Text Tokens    ──────┘                           │
                  (FINDINGS+HISTORY)                                                             ├─→ [1024+512+32=1568-d] → MLP → 2 logits
[WBC Count]    →  Scalar MLP         →  32-d Meta Embedding ──────────────────────────────────┘
```

## Key Differences vs Phase 3 (16 features)
| Aspect                | Phase 3 (Full)            | Phase 3c (WBC Only)       |
|-----------------------|---------------------------|---------------------------|
| Clinical features     | 16 (all labs+vitals+demo) | **1 (WBC only)**          |
| Meta encoder          | MLP(16→128→128→64)        | **MLP(1→32→32→32)**       |
| Fused dim             | 1600                      | **1568**                  |
| EHR missingness risk  | High (CRP, Albumin, etc.) | **Low (CBC always done)** |
| Clinical interpretability | Complex              | **Direct: WBC = Infection marker** |

In [4]:
# --- CELL 0: Imports & Reproducibility ---
import os, random, warnings, json, copy, re
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from transformers import AutoTokenizer, AutoModel
import torchxrayvision as xrv

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, accuracy_score, f1_score
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
print(f'PyTorch: {torch.__version__}')
print('✅ Imports done.')

Device : cuda
PyTorch: 2.5.1+cu121
✅ Imports done.


In [5]:
# --- CELL 1: Configuration & Paths (SCALEUP VERSION - 11 FEATURES) ---
import os

MAIN_DIR     = r'C:\2026\PneumoFusionNet\mimic\main'
DATASET_DIR  = os.path.join(MAIN_DIR, 'dataset')
BASE_DIR     = MAIN_DIR

# Input Data — Scaleup single merged CSV (~3,763 samples)
PAIRED_CSV   = os.path.join(DATASET_DIR, 'phase3_paired_scaleup_final.csv')
TEXT_CSV     = PAIRED_CSV
CLINICAL_CSV = PAIRED_CSV
BBOX_CSV     = os.path.join(MAIN_DIR, 'outputs', 'Phase_1.1v4_PA_crossval_scaleup', 'lung_bboxes.csv')

# Pretrained Weights (Scaleup Fold 5 & Phase 2v2 Scaleup)
P1_CKPT      = os.path.join(MAIN_DIR, 'outputs', 'Phase_1.1v4_PA_crossval_scaleup', 'best_model_fold5.pth')
P2V2_CKPT    = os.path.join(MAIN_DIR, 'outputs', 'Phase_2v2_Scaleup', 'best_v2_model.pth')

,# Output Directory
SAVE_DIR     = os.path.join(MAIN_DIR, 'outputs', 'Phase_3_triple_fusion_scaleup_11features')
os.makedirs(SAVE_DIR, exist_ok=True)

# Model Hyperparameters
IMG_SIZE       = 224
MAX_TEXT_LEN   = 256
CLINBERT_MODEL = 'emilyalsentzer/Bio_ClinicalBERT'
IMG_FEAT_DIM   = 1024
TXT_FEAT_DIM   = 768
ATTN_DIM       = 512
ATTN_HEADS     = 8
META_HIDDEN    = 128
META_OUT_DIM   = 64

# Training Hyperparameters
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
SEED        = 42
BATCH_SIZE  = 16
LR_FUSION   = 2e-4
LR_BERT     = 1e-5
LR_META     = 1e-3
EPOCHS      = 40
PATIENCE    = 8
FOCAL_GAMMA = 2.0
MIXUP_ALPHA = 0.2
CLASSES     = ['NORMAL', 'PNEUMONIA']
MEAN = [0.5020]; STD = [0.2703]

# WBC-ONLY ablation study (1 feature)
CLINICAL_FEATURES   = ['wbc']
N_CLINICAL_FEATURES = len(CLINICAL_FEATURES)  # = 1
print(f'Paired CSV     : {os.path.exists(PAIRED_CSV)} -> {PAIRED_CSV}')
print(f'BBox CSV       : {os.path.exists(BBOX_CSV)}')
print(f'P1 Checkpoint  : {os.path.exists(P1_CKPT)}')
print(f'P2v2 Checkpoint: {os.path.exists(P2V2_CKPT)}')
print(f'Save Dir       : {SAVE_DIR}')
print(f'Clinical Feats : {N_CLINICAL_FEATURES} {CLINICAL_FEATURES}')

Paired CSV     : True -> C:\2026\PneumoFusionNet\mimic\main\dataset\phase3_paired_scaleup_final.csv
BBox CSV       : True
P1 Checkpoint  : True
P2v2 Checkpoint: True
Save Dir       : C:\2026\PneumoFusionNet\mimic\main\outputs\Phase_3_triple_fusion_scaleup_11features
Clinical Feats : 1 ['wbc']


In [7]:
# --- CELL 2: Data Loading & Merging (SCALEUP) ---

def resolve_path(p):
    if pd.isna(p) or str(p).strip() == '': return ''
    p = str(p).replace('/', os.sep).replace('\\', os.sep)
    if os.path.isabs(p): return p
    return os.path.join(DATASET_DIR, p)

text_df = pd.read_csv(PAIRED_CSV)
text_df['image_path']  = text_df['image_path'].apply(resolve_path)
text_df['report_path'] = text_df['report_path'].apply(resolve_path)

if 'raw_report' in text_df.columns:
    text_df['report_no_impression'] = text_df['raw_report'].fillna('').astype(str)
elif 'report_no_impression' in text_df.columns:
    text_df['report_no_impression'] = text_df['report_no_impression'].fillna('').astype(str)
else:
    def read_report(path):
        try:
            with open(path, encoding='utf-8', errors='ignore') as f: return f.read().strip()
        except: return ''
    text_df['report_no_impression'] = text_df['report_path'].apply(read_report)

clinical_df = text_df.copy()
print(f'Text CSV rows     : {len(text_df)}')
print(f'Clinical CSV rows : {len(clinical_df)}')
print(f'Images found      : {text_df["image_path"].apply(os.path.exists).sum()}/{len(text_df)}')
print(f'Reports found     : {text_df["report_path"].apply(os.path.exists).sum()}/{len(text_df)}')


Text CSV rows     : 3763
Clinical CSV rows : 3763
Images found      : 3763/3763
Reports found     : 3763/3763


In [8]:
# --- CELL 3: Text Extraction (FINDINGS + HISTORY) — same as Phase 2v2 & Phase 3 ---

LEAKAGE_RE = re.compile(
    r'\bpneumonia\b|\bpneumonic\b|\bno[ -]acute[ -]\w+|\bno finding\w*'
    r'|\bcompatible with\b|\bconsistent with\b|\bnormal study\b|\bno significant\b',
    re.IGNORECASE
)

def extract_rich_text(text):
    parts = []
    h = re.search(r'HISTORY[:\s]+(.*?)(?=FINDINGS|TECHNIQUE|COMPARISON|\n\n|\Z)',
                  text, re.DOTALL | re.IGNORECASE)
    if h: parts.append(h.group(1).strip())
    f = re.search(r'FINDINGS[:\s]+(.*?)(?=IMPRESSION|CONCLUSION|\n\n|\Z)',
                  text, re.DOTALL | re.IGNORECASE)
    if f: parts.append(f.group(1).strip())
    result = ' '.join(parts).strip()
    if not result: result = text[:512]
    result = LEAKAGE_RE.sub('[REDACTED]', result)
    return result

def read_report(path):
    try:
        with open(path, encoding='utf-8', errors='ignore') as f:
            return f.read().strip()
    except: return ''

has_report = text_df['report_path'].apply(os.path.exists)
print(f'Reports on disk: {has_report.sum()}/{len(text_df)}')

text_df['full_report'] = text_df['report_path'].apply(read_report)
text_df['report_rich'] = text_df['full_report'].apply(extract_rich_text)
print(f'Avg words (rich): {text_df["report_rich"].str.split().str.len().mean():.1f}')
print('✅ Text extraction done (anti-leakage: IMPRESSION removed, keywords redacted).')

Reports on disk: 3763/3763
Avg words (rich): 50.3
✅ Text extraction done (anti-leakage: IMPRESSION removed, keywords redacted).


In [10]:
# Validate clinical features
CLINICAL_FEATURES = [c for c in CLINICAL_FEATURES if c in df.columns]
N_CLINICAL_FEATURES = len(CLINICAL_FEATURES)
print(f'\nUsing {N_CLINICAL_FEATURES} clinical features: {CLINICAL_FEATURES}')

# Fill NaN with median before split
for col in CLINICAL_FEATURES:
    if df[col].dtype in [float, int, 'float64', 'int64']:
        median_val = df[col].median()
        if pd.isna(median_val): median_val = 0.0
        df[col] = df[col].fillna(median_val)
    else:
        df[col] = df[col].fillna(0)

print(f'Missing values after fill: {df[CLINICAL_FEATURES].isnull().sum().sum()} (must be 0)')



Using 0 clinical features: []
Missing values after fill: 0.0 (must be 0)


In [13]:
# --- CELL 5: Train / Val / Test Split (70/15/15, stratified) ---
train_val_df, test_df = train_test_split(
    df, test_size=0.15, stratify=df['label'], random_state=SEED)
val_frac = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
train_df, val_df = train_test_split(
    train_val_df, test_size=val_frac, stratify=train_val_df['label'], random_state=SEED)

for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    n0 = (split['label']==0).sum(); n1 = (split['label']==1).sum()
    print(f'{name:5s}: {len(split):5d}  (Normal={n0}, Pneumonia={n1})')

# Fit scaler ONLY on training data to avoid data leakage
scaler = StandardScaler()
train_df = train_df.copy()
val_df   = val_df.copy()
test_df  = test_df.copy()

# Resolve the WBC feature robustly for merged CSVs (handles wbc / wbc_x / wbc_y)
for split_name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    if 'wbc' in split_df.columns:
        wbc_col = 'wbc'
    elif 'wbc_x' in split_df.columns and 'wbc_y' in split_df.columns:
        split_df['wbc'] = split_df['wbc_x'].combine_first(split_df['wbc_y'])
        wbc_col = 'wbc'
    else:
        wbc_candidates = [c for c in split_df.columns if 'wbc' in c.lower()]
        if not wbc_candidates:
            raise KeyError(f'No WBC column found in {split_name}_df')
        split_df['wbc'] = split_df[wbc_candidates[0]]
        wbc_col = 'wbc'

    split_df[wbc_col] = split_df[wbc_col].astype(float)
    median_val = split_df[wbc_col].median()
    if pd.isna(median_val):
        median_val = 0.0
    split_df[wbc_col] = split_df[wbc_col].fillna(median_val)

# Use the canonical clinical feature name
CLINICAL_FEATURES = ['wbc']
train_df[CLINICAL_FEATURES] = scaler.fit_transform(train_df[CLINICAL_FEATURES].astype(float))
val_df[CLINICAL_FEATURES]   = scaler.transform(val_df[CLINICAL_FEATURES].astype(float))
test_df[CLINICAL_FEATURES]  = scaler.transform(test_df[CLINICAL_FEATURES].astype(float))
val_df[CLINICAL_FEATURES]   = scaler.transform(val_df[CLINICAL_FEATURES].astype(float))
test_df[CLINICAL_FEATURES]  = scaler.transform(test_df[CLINICAL_FEATURES].astype(float))
print('\nClinical features standardised (fit on train only). No data leakage.')

Train:  2807  (Normal=1369, Pneumonia=1438)
Val  :   602  (Normal=293, Pneumonia=309)
Test :   602  (Normal=294, Pneumonia=308)

Clinical features standardised (fit on train only). No data leakage.


In [15]:
# --- CELL 5: Train / Val / Test Split (70/15/15, stratified) ---
train_val_df, test_df = train_test_split(
    df, test_size=0.15, stratify=df['label'], random_state=SEED)
val_frac = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
train_df, val_df = train_test_split(
    train_val_df, test_size=val_frac, stratify=train_val_df['label'], random_state=SEED)

for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    n0 = (split['label']==0).sum(); n1 = (split['label']==1).sum()
    print(f'{name:5s}: {len(split):5d}  (Normal={n0}, Pneumonia={n1})')

# Fit scaler ONLY on training data to avoid data leakage
scaler = StandardScaler()
train_df = train_df.copy()
val_df   = val_df.copy()
test_df  = test_df.copy()

# Resolve the WBC feature robustly (handles wbc / wbc_x / wbc_y)
for split_name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    if 'wbc' in split_df.columns:
        wbc_col = 'wbc'
    elif 'wbc_x' in split_df.columns and 'wbc_y' in split_df.columns:
        split_df['wbc'] = split_df['wbc_x'].combine_first(split_df['wbc_y'])
        wbc_col = 'wbc'
    else:
        wbc_candidates = [c for c in split_df.columns if 'wbc' in c.lower()]
        if not wbc_candidates:
            raise KeyError(f'No WBC column found in {split_name}_df')
        split_df['wbc'] = split_df[wbc_candidates[0]]
        wbc_col = 'wbc'

    split_df[wbc_col] = split_df[wbc_col].astype(float)
    median_val = split_df[wbc_col].median()
    if pd.isna(median_val):
        median_val = 0.0
    split_df[wbc_col] = split_df[wbc_col].fillna(median_val)

# Scale
CLINICAL_FEATURES = ['wbc']
train_df[CLINICAL_FEATURES] = scaler.fit_transform(train_df[CLINICAL_FEATURES].astype(float))
val_df[CLINICAL_FEATURES]   = scaler.transform(val_df[CLINICAL_FEATURES].astype(float))
test_df[CLINICAL_FEATURES]  = scaler.transform(test_df[CLINICAL_FEATURES].astype(float))
print(f'\nWBC after scaling — mean={train_df["wbc"].mean():.4f}, std={train_df["wbc"].std():.4f}')
print('✅ Split + StandardScaler done. No data leakage.')

Train:  2807  (Normal=1369, Pneumonia=1438)
Val  :   602  (Normal=293, Pneumonia=309)
Test :   602  (Normal=294, Pneumonia=308)

WBC after scaling — mean=-0.0000, std=1.0002
✅ Split + StandardScaler done. No data leakage.


In [20]:
# --- CELL 6: Bbox Lookup & Triple Dataset Class ---
bbox_df     = pd.read_csv(BBOX_CSV)
bbox_lookup = bbox_df.set_index('image_path').to_dict('index')
print(f'Bbox entries: {len(bbox_lookup)}')

class TripleModalCXRDataset(Dataset):
    """Dataset returning (image, text_ids, text_mask, clinical_features, label)"""
    def __init__(self, df, bbox_lookup, tokenizer, img_transform,
                 clinical_features, max_len=MAX_TEXT_LEN):
        self.df               = df.reset_index(drop=True)
        self.bbox_lookup      = bbox_lookup
        self.tokenizer        = tokenizer
        self.transform        = img_transform
        self.clinical_features = clinical_features
        self.max_len          = max_len
        self.clahe            = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # --- IMAGE ---
        img = cv2.imread(row.image_path, cv2.IMREAD_GRAYSCALE)
        if img is None: img = np.zeros((224,224), dtype=np.uint8)
        img = self.clahe.apply(img)
        bb  = self.bbox_lookup.get(row.image_path, None)
        if bb and bb.get('x_max',0)>0:
            img = img[bb['y_min']:bb['y_max'], bb['x_min']:bb['x_max']]
        img = self.transform(Image.fromarray(img))

        # --- TEXT ---
        enc = self.tokenizer(
            row.report_rich, max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )

        # --- CLINICAL METADATA ---
        meta = torch.tensor(
            [float(row[c]) for c in self.clinical_features],
            dtype=torch.float32
        )

        return (
            img,
            enc['input_ids'].squeeze(0),
            enc['attention_mask'].squeeze(0),
            meta,
            int(row.label)
        )

    def set_transform(self, t): self.transform = t

val_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])
train_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(8),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])
print('Dataset class ready.')

Bbox entries: 3744
Dataset class ready.


In [21]:
# --- CELL 8: Model Architecture ---

# ──── CBAM (identical to Phase 1, 2, 3) ────
class ChannelAttention(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        self.avg=nn.AdaptiveAvgPool2d(1); self.max=nn.AdaptiveMaxPool2d(1)
        self.mlp=nn.Sequential(nn.Conv2d(c,c//r,1,bias=False),nn.ReLU(True),nn.Conv2d(c//r,c,1,bias=False))
        self.sig=nn.Sigmoid()
    def forward(self,x): return x*self.sig(self.mlp(self.avg(x))+self.mlp(self.max(x)))

class SpatialAttention(nn.Module):
    def __init__(self,k=7):
        super().__init__()
        self.conv=nn.Conv2d(2,1,k,padding=k//2,bias=False); self.sig=nn.Sigmoid()
    def forward(self,x):
        return x*self.sig(self.conv(torch.cat([x.mean(1,keepdim=True),x.max(1,keepdim=True)[0]],1)))

class CBAM(nn.Module):
    def __init__(self,c,r=16):
        super().__init__()
        self.ca=ChannelAttention(c,r); self.sa=SpatialAttention()
    def forward(self,x): return self.sa(self.ca(x))

# ──── Image Encoder (frozen DenseNet-121+CBAM) ────
class ImageEncoder(nn.Module):
    def __init__(self, ckpt):
        super().__init__()
        xrv_m=xrv.models.DenseNet(weights='densenet121-res224-all')
        self.features=xrv_m.features; self.cbam=CBAM(1024); self.pool=nn.AdaptiveAvgPool2d(1)
        sd={k:v for k,v in torch.load(ckpt,map_location='cpu').items()
            if k.startswith('features.') or k.startswith('cbam.')}
        miss,unexp=self.load_state_dict(sd,strict=False)
        print(f'[ImageEncoder] loaded {len(sd)} keys | missing={len(miss)} unexpected={len(unexp)}')
        for p in self.parameters(): p.requires_grad=False
    def forward(self,x):
        f=F.relu(self.features(x),True); f=self.cbam(f)
        return self.pool(f).flatten(1)       # (B, 1024)

# ──── Text Encoder: ClinicalBERT (last 2 layers unfrozen) ────
class TextEncoder(nn.Module):
    def __init__(self, model_name=CLINBERT_MODEL):
        super().__init__()
        self.bert=AutoModel.from_pretrained(model_name)
        for name,p in self.bert.named_parameters():
            if 'encoder.layer.10' in name or 'encoder.layer.11' in name or 'pooler' in name:
                p.requires_grad=True
            else:
                p.requires_grad=False
        trainable=sum(p.numel() for p in self.bert.parameters() if p.requires_grad)
        print(f'[TextEncoder] Unfrozen BERT params: {trainable:,}')
    def forward(self,input_ids,attention_mask):
        out=self.bert(input_ids=input_ids,attention_mask=attention_mask)
        return out.last_hidden_state   # (B, seq_len, 768)

# ──── WBC Metadata Encoder (minimal MLP for single scalar input) ────
class MetadataEncoder(nn.Module):
    """
    Lightweight MLP for WBC-only input: 1 → 32 → 32 → 32-d embedding.
    Smaller than Phase 3 (16→128→128→64) since input is just 1 scalar.
    """
    def __init__(self, in_dim=N_CLINICAL_FEATURES, hidden=META_HIDDEN, out_dim=META_OUT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, out_dim),
            nn.ReLU()
        )
        trainable = sum(p.numel() for p in self.net.parameters())
        print(f'[MetadataEncoder (WBC-only)] in_dim={in_dim} → out_dim={out_dim}')
        print(f'[MetadataEncoder] Trainable params: {trainable:,}')
    def forward(self, x):
        return self.net(x)  # (B, META_OUT_DIM=32)

# ──── WBC Triple Fusion Net ────
class TripleFusionNet(nn.Module):
    """
    WBC-Only Triple Fusion:
      img_feat   (B, 1024) from DenseNet+CBAM
      txt_tokens (B, seq, 768) from ClinicalBERT
      meta       (B, 32) from WBC scalar MLP

    Cross-Attn: image queries text → (B, 512)
    Concat: [img(1024) + cross_attn(512) + meta(32)] = 1568-d
    Classifier: LayerNorm → Dropout → Linear chain → 2 logits
    """
    def __init__(self,
                 img_dim=IMG_FEAT_DIM,
                 txt_dim=TXT_FEAT_DIM,
                 attn_dim=ATTN_DIM,
                 heads=ATTN_HEADS,
                 meta_out=META_OUT_DIM):
        super().__init__()
        # Cross-attention (same as Phase 2v2 and Phase 3)
        self.query_proj = nn.Linear(img_dim,  attn_dim)
        self.key_proj   = nn.Linear(txt_dim,  attn_dim)
        self.val_proj   = nn.Linear(txt_dim,  attn_dim)
        self.cross_attn = nn.MultiheadAttention(attn_dim, heads, batch_first=True, dropout=0.1)
        self.norm1      = nn.LayerNorm(attn_dim)

        # Classifier (1024 + 512 + 32 = 1568-d input)
        fused_dim = img_dim + attn_dim + meta_out  # 1568
        self.classifier = nn.Sequential(
            nn.LayerNorm(fused_dim),
            nn.Dropout(0.4),
            nn.Linear(fused_dim, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 2)
        )
        print(f'[TripleFusionNet (WBC-only)] fused_dim={fused_dim}')
        print(f'  img={img_dim} + cross_attn={attn_dim} + wbc_meta={meta_out}')

    def forward(self, img_feat, txt_tokens, meta_feat):
        q = self.query_proj(img_feat).unsqueeze(1)   # (B, 1, 512)
        k = self.key_proj(txt_tokens)                 # (B, seq, 512)
        v = self.val_proj(txt_tokens)                 # (B, seq, 512)
        attended, _ = self.cross_attn(q, k, v)        # (B, 1, 512)
        attended = self.norm1(attended.squeeze(1))     # (B, 512)
        fused = torch.cat([img_feat, attended, meta_feat], dim=1)  # (B, 1568)
        return self.classifier(fused)

print('✅ All model classes defined.')

✅ All model classes defined.


In [22]:
# --- CELL 9: Loss Function & Triple Mixup Utilities ---

class FocalLoss(nn.Module):
    def __init__(self, gamma=FOCAL_GAMMA, weight=None):
        super().__init__()
        self.gamma=gamma; self.weight=weight
    def forward(self, logits, labels):
        ce  = F.cross_entropy(logits, labels, weight=self.weight, reduction='none')
        pt  = torch.exp(-ce)
        loss= ((1-pt)**self.gamma * ce).mean()
        return loss

def mixup_triple(img_feat, txt_tokens, meta_feat, labels, alpha=MIXUP_ALPHA):
    """Mixup on all three embedding branches simultaneously."""
    if alpha <= 0: return img_feat, txt_tokens, meta_feat, labels, labels, 1.0
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(img_feat.size(0), device=img_feat.device)
    img_mix  = lam * img_feat   + (1-lam) * img_feat[idx]
    txt_mix  = lam * txt_tokens + (1-lam) * txt_tokens[idx]
    meta_mix = lam * meta_feat  + (1-lam) * meta_feat[idx]
    return img_mix, txt_mix, meta_mix, labels, labels[idx], lam

def mixup_criterion(criterion, logits, labels_a, labels_b, lam):
    return lam * criterion(logits, labels_a) + (1-lam) * criterion(logits, labels_b)

print('✅ Focal Loss and Triple Mixup ready.')

✅ Focal Loss and Triple Mixup ready.


In [23]:
# --- CELL 10: Instantiate Models, Load Phase 2v2 Weights, Build DataLoaders ---

print('Loading ClinicalBERT tokenizer...')
tokenizer     = AutoTokenizer.from_pretrained(CLINBERT_MODEL)
text_encoder  = TextEncoder(CLINBERT_MODEL).to(DEVICE)

print('\nLoading Phase-1 Image Encoder (Fold 2)...')
image_encoder = ImageEncoder(P1_CKPT).to(DEVICE)

print(f'\nBuilding WBC MetadataEncoder (in_dim={N_CLINICAL_FEATURES})...')
meta_encoder  = MetadataEncoder(in_dim=N_CLINICAL_FEATURES,
                                hidden=META_HIDDEN,
                                out_dim=META_OUT_DIM).to(DEVICE)

print('\nBuilding TripleFusionNet (WBC-Only)...')
fusion_model  = TripleFusionNet(meta_out=META_OUT_DIM).to(DEVICE)

# ── Load Phase 2v2 weights into text + cross-attention components ──
print(f'\nLoading Phase 2v2 checkpoint: {os.path.exists(P2V2_CKPT)}')
if os.path.exists(P2V2_CKPT):
    p2v2_state = torch.load(P2V2_CKPT, map_location='cpu')

    if 'text' in p2v2_state:
        miss, unexp = text_encoder.load_state_dict(p2v2_state['text'], strict=False)
        print(f'  [TextEncoder] Loaded P2v2 weights | missing={len(miss)} unexpected={len(unexp)}')

    if 'fusion' in p2v2_state:
        fusion_p2v2 = p2v2_state['fusion']
        ca_keys = {k:v for k,v in fusion_p2v2.items()
                   if any(k.startswith(pfx) for pfx in
                          ['query_proj','key_proj','val_proj','cross_attn','norm1'])}
        miss2, unexp2 = fusion_model.load_state_dict(ca_keys, strict=False)
        print(f'  [FusionNet] Loaded {len(ca_keys)} cross-attn keys from P2v2')
        print(f'  (Classifier head re-init: 1536-d → 1568-d for WBC triple fusion)')
else:
    print('  WARNING: P2v2 checkpoint not found. Training from scratch!')

# Parameter count
print(f'\nTrainable parameter count:')
print(f'  Fusion Net     : {sum(p.numel() for p in fusion_model.parameters() if p.requires_grad):,}')
print(f'  BERT (unfrozen): {sum(p.numel() for p in text_encoder.parameters() if p.requires_grad):,}')
print(f'  WBC Meta MLP   : {sum(p.numel() for p in meta_encoder.parameters() if p.requires_grad):,}')

# ── DataLoaders ──
train_ds = TripleModalCXRDataset(train_df, bbox_lookup, tokenizer, train_tfm, CLINICAL_FEATURES)
val_ds   = TripleModalCXRDataset(val_df,   bbox_lookup, tokenizer, val_tfm,   CLINICAL_FEATURES)
test_ds  = TripleModalCXRDataset(test_df,  bbox_lookup, tokenizer, val_tfm,   CLINICAL_FEATURES)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
print(f'\n✅ DataLoaders ready | Train={len(train_loader)} | Val={len(val_loader)} | Test={len(test_loader)} batches')

Loading ClinicalBERT tokenizer...
[TextEncoder] Unfrozen BERT params: 14,766,336

Loading Phase-1 Image Encoder (Fold 2)...
[ImageEncoder] loaded 728 keys | missing=3 unexpected=3

Building WBC MetadataEncoder (in_dim=0)...
[MetadataEncoder (WBC-only)] in_dim=0 → out_dim=64
[MetadataEncoder] Trainable params: 24,896

Building TripleFusionNet (WBC-Only)...
[TripleFusionNet (WBC-only)] fused_dim=1600
  img=1024 + cross_attn=512 + wbc_meta=64

Loading Phase 2v2 checkpoint: True
  [TextEncoder] Loaded P2v2 weights | missing=0 unexpected=0
  [FusionNet] Loaded 12 cross-attn keys from P2v2
  (Classifier head re-init: 1536-d → 1568-d for WBC triple fusion)

Trainable parameter count:
  Fusion Net     : 3,252,738
  BERT (unfrozen): 14,766,336
  WBC Meta MLP   : 24,896

✅ DataLoaders ready | Train=176 | Val=38 | Test=38 batches


In [24]:
# --- CELL 11: Training Loop ---

def eval_epoch(fusion, img_enc, txt_enc, meta_enc, loader, criterion, device):
    fusion.eval(); img_enc.eval(); txt_enc.eval(); meta_enc.eval()
    total_loss, all_probs, all_labels = 0.0, [], []
    with torch.no_grad():
        for imgs,ids,masks,meta,labels in loader:
            imgs   = imgs.to(device)
            ids    = ids.to(device)
            masks  = masks.to(device)
            meta   = meta.to(device)
            labels = labels.to(device)

            img_f   = img_enc(imgs)
            txt_t   = txt_enc(ids, masks)
            meta_f  = meta_enc(meta)
            logits  = fusion(img_f, txt_t, meta_f)

            loss  = criterion(logits, labels)
            probs = F.softmax(logits, dim=1)[:,1].cpu().numpy()
            total_loss += loss.item()*labels.size(0)
            all_probs.extend(probs); all_labels.extend(labels.cpu().tolist())

    acc = accuracy_score(all_labels,[1 if p>=0.5 else 0 for p in all_probs])
    auc = roc_auc_score(all_labels, all_probs) if len(set(all_labels))>1 else 0.5
    return total_loss/len(all_labels), acc, auc, all_probs, all_labels

def find_optimal_threshold(labels, probs):
    fpr,tpr,thresh=roc_curve(labels,probs)
    return float(thresh[np.argmax(tpr-fpr)])

def find_clinical_threshold(labels, probs, target_sens=0.90):
    fpr,tpr,thresh=roc_curve(labels,probs)
    for t,s in zip(thresh, tpr):
        if s >= target_sens:
            return float(t)
    return float(thresh[np.argmax(tpr-fpr)])

# Class weights
n0=(train_df['label']==0).sum(); n1=(train_df['label']==1).sum()
weight=torch.tensor([n1/(n0+n1), n0/(n0+n1)], dtype=torch.float).to(DEVICE)
criterion_focal = FocalLoss(gamma=FOCAL_GAMMA, weight=weight)

# 3 separate learning rates
optimizer = torch.optim.AdamW([
    {'params': text_encoder.parameters(),  'lr': LR_BERT,   'weight_decay': 1e-4},
    {'params': fusion_model.parameters(),  'lr': LR_FUSION, 'weight_decay': 1e-4},
    {'params': meta_encoder.parameters(),  'lr': LR_META,   'weight_decay': 1e-3},
])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_auc, best_state, patience_cnt = 0.0, None, 0
history = {'train_loss':[], 'val_loss':[], 'train_auc':[], 'val_auc':[]}

print('='*55)
print('  Phase 3c (WBC-Only) Training')
print('='*55)

for epoch in range(1, EPOCHS+1):
    # ── Train ──
    fusion_model.train(); image_encoder.eval(); text_encoder.train(); meta_encoder.train()
    ep_loss, ep_probs, ep_labels = 0.0, [], []

    for imgs,ids,masks,meta,labels in train_loader:
        imgs   = imgs.to(DEVICE)
        ids    = ids.to(DEVICE)
        masks  = masks.to(DEVICE)
        meta   = meta.to(DEVICE)
        labels = labels.to(DEVICE)

        with torch.no_grad():
            img_f = image_encoder(imgs)       # frozen
        txt_t  = text_encoder(ids, masks)
        meta_f = meta_encoder(meta)           # WBC scalar → 32-d

        img_m, txt_m, meta_m, la, lb, lam = mixup_triple(img_f, txt_t, meta_f, labels)

        optimizer.zero_grad()
        logits = fusion_model(img_m, txt_m, meta_m)
        loss   = mixup_criterion(criterion_focal, logits, la, lb, lam)
        loss.backward()

        all_params = (list(fusion_model.parameters()) +
                      list(text_encoder.parameters()) +
                      list(meta_encoder.parameters()))
        nn.utils.clip_grad_norm_(all_params, 1.0)
        optimizer.step()

        probs = F.softmax(logits.detach(), dim=1)[:,1].cpu().numpy()
        ep_loss += loss.item()*labels.size(0)
        ep_probs.extend(probs); ep_labels.extend(labels.cpu().tolist())

    scheduler.step()
    tr_loss = ep_loss/len(ep_labels)
    tr_auc  = roc_auc_score(ep_labels, ep_probs) if len(set(ep_labels))>1 else 0.5

    # ── Val ──
    val_loss,val_acc,val_auc,_,_ = eval_epoch(
        fusion_model, image_encoder, text_encoder, meta_encoder,
        val_loader, criterion_focal, DEVICE)

    history['train_loss'].append(tr_loss); history['val_loss'].append(val_loss)
    history['train_auc'].append(tr_auc);   history['val_auc'].append(val_auc)

    marker=''
    if val_auc > best_auc:
        best_auc=val_auc
        best_state=copy.deepcopy({
            'fusion': fusion_model.state_dict(),
            'text':   text_encoder.state_dict(),
            'meta':   meta_encoder.state_dict()
        })
        torch.save(best_state, os.path.join(SAVE_DIR,'best_p3c_model.pth'))
        patience_cnt=0; marker=' ✅ best'
    else:
        patience_cnt+=1

    if epoch%5==0 or marker:
        print(f'Ep {epoch:03d} | tr_loss={tr_loss:.4f} tr_auc={tr_auc:.4f} '
              f'| val_auc={val_auc:.4f} val_acc={val_acc:.3f}{marker}')
    if patience_cnt>=PATIENCE:
        print(f'Early stop at epoch {epoch}.'); break

print(f'\nBest Val AUC: {best_auc:.4f}')

  Phase 3c (WBC-Only) Training


RuntimeError: mat1 and mat2 shapes cannot be multiplied (16x1 and 0x128)

In [ ]:
# --- CELL 12: Test Set Evaluation ---
fusion_model.load_state_dict(best_state['fusion'])
text_encoder.load_state_dict(best_state['text'])
meta_encoder.load_state_dict(best_state['meta'])

_,test_acc,test_auc,test_probs,test_labels = eval_epoch(
    fusion_model, image_encoder, text_encoder, meta_encoder,
    test_loader, criterion_focal, DEVICE)

thresh_youden   = find_optimal_threshold(test_labels, test_probs)
thresh_clinical = find_clinical_threshold(test_labels, test_probs, target_sens=0.90)

preds_def      = [1 if p>=0.5            else 0 for p in test_probs]
preds_youden   = [1 if p>=thresh_youden  else 0 for p in test_probs]
preds_clinical = [1 if p>=thresh_clinical else 0 for p in test_probs]

def compute_metrics(labels, preds, name, thresh):
    cm = confusion_matrix(labels, preds)
    s  = cm[1,1]/(cm[1,0]+cm[1,1]) if (cm[1,0]+cm[1,1])>0 else 0
    sp = cm[0,0]/(cm[0,0]+cm[0,1]) if (cm[0,0]+cm[0,1])>0 else 0
    ac = (cm[0,0]+cm[1,1])/cm.sum()
    f1 = f1_score(labels, preds)
    print(f'{name} (thresh={thresh:.3f}): Acc={ac:.3f} | F1={f1:.3f} | Sens={s:.3f} | Spec={sp:.3f}')
    return s, sp, ac, f1

print('='*60)
print('  PHASE 3c (WBC-ONLY) — TEST RESULTS')
print('='*60)
print(f'Test AUC: {test_auc:.4f}  |  Best Val AUC: {best_auc:.4f}\n')
s_d,sp_d,ac_d,f1_d     = compute_metrics(test_labels, preds_def,      'Default   ', 0.5)
s_y,sp_y,ac_y,f1_y     = compute_metrics(test_labels, preds_youden,   'Youden-J  ', thresh_youden)
s_c,sp_c,ac_c,f1_c     = compute_metrics(test_labels, preds_clinical, 'Clinical  ', thresh_clinical)

In [ ]:
# --- CELL 13: Confusion Matrices ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, preds, title in zip(axes,
    [preds_def, preds_youden, preds_clinical],
    ['Default (0.5)', f'Youden-J ({thresh_youden:.3f})', f'Clinical ({thresh_clinical:.3f})']):
    cm_arr = confusion_matrix(test_labels, preds)
    sns.heatmap(cm_arr, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=CLASSES, yticklabels=CLASSES)
    ax.set_title(f'Phase 3c WBC-Only | {title}', fontsize=13)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.suptitle(f'Phase 3c WBC-Only | Test AUC={test_auc:.4f}', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'p3c_confusion_matrices.png'), bbox_inches='tight', dpi=150)
plt.show()
print('✅ Saved confusion matrices.')

In [ ]:
# --- CELL 14: ROC Curve ---
fpr, tpr, _ = roc_curve(test_labels, test_probs)

plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, lw=2.5, color='crimson', label=f'Phase 3c WBC-Only (AUC={test_auc:.4f})')
plt.plot([0,1],[0,1],'k--', alpha=0.5)
plt.axvline(1-sp_y, ls=':', color='blue', alpha=0.6, label=f'Youden-J point (Sens={s_y:.3f}, Spec={sp_y:.3f})')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Phase 3c WBC-Only — ROC Curve', fontsize=14)
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'p3c_roc_curve.png'), dpi=150)
plt.show()
print('✅ Saved ROC curve.')

In [ ]:
# --- CELL 15: Training Curves ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_auc'], label='Train AUC', color='royalblue', lw=2)
axes[0].plot(history['val_auc'],   label='Val AUC',   color='darkorange', lw=2)
axes[0].axhline(0.949,  ls='--', color='green',  alpha=0.7, label='Phase 2v2 (0.949)')
axes[0].axhline(0.9841, ls=':',  color='purple', alpha=0.7, label='Phase 3 16-feat (0.984)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('AUC')
axes[0].set_title('AUC over Epochs'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_loss'], label='Train Loss', color='royalblue', lw=2)
axes[1].plot(history['val_loss'],   label='Val Loss',   color='darkorange', lw=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Focal Loss')
axes[1].set_title('Loss over Epochs'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Phase 3c (WBC-Only) — Training History', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'p3c_training_curves.png'), dpi=150)
plt.show()
print('✅ Saved training curves.')

In [ ]:
# --- CELL 16: Full Pipeline Comparison Chart ---
# Compare all phases including WBC-only result vs baseline

phases = [
    {'name': 'Phase 1\nImage only\n(DenseNet v4)', 'auc': 0.845, 'sens': 79.5, 'spec': 76.0,  'color': '#4477AA'},
    {'name': 'Phase 2v1\nImg+Text\n(Concat)',       'auc': 0.911, 'sens': 80.6, 'spec': 89.4,  'color': '#66CCEE'},
    {'name': 'Phase 2v2\nImg+Text\n(CrossAttn)',    'auc': 0.949, 'sens': 91.4, 'spec': 86.3,  'color': '#228833'},
    {'name': 'Phase 3\nImg+Text\n+16 feats',        'auc': 0.984, 'sens': 94.9, 'spec': 94.6,  'color': '#EE6677'},
    {'name': 'Phase 3c\nImg+Text\n+WBC only ✨',    'auc': round(test_auc, 4),
                                                     'sens': round(s_y*100, 1),
                                                     'spec': round(sp_y*100, 1), 'color': '#AA3377'},
]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
names  = [p['name']  for p in phases]
colors = [p['color'] for p in phases]

# ── AUC ──
ax = axes[0]
bars = ax.bar(names, [p['auc'] for p in phases], color=colors, edgecolor='black', alpha=0.85)
ax.set_ylim(0.75, 1.01)
ax.set_ylabel('Test AUC', fontsize=12); ax.set_title('Test AUC Comparison', fontsize=13)
ax.grid(axis='y', alpha=0.3)
for i,p in enumerate(phases):
    ax.text(i, p['auc']+0.003, f"{p['auc']:.3f}", ha='center', fontsize=10, fontweight='bold')
# Highlight WBC-only bar
bars[-1].set_edgecolor('gold'); bars[-1].set_linewidth(2.5)

# ── Sensitivity ──
ax = axes[1]
bars = ax.bar(names, [p['sens'] for p in phases], color=colors, edgecolor='black', alpha=0.85)
ax.set_ylim(60, 102)
ax.axhline(90, ls='--', color='red', alpha=0.6, label='90% clinical target')
ax.set_ylabel('Sensitivity (%)', fontsize=12); ax.set_title('Sensitivity (Youden-J)', fontsize=13)
ax.legend(); ax.grid(axis='y', alpha=0.3)
for i,p in enumerate(phases):
    ax.text(i, p['sens']+0.5, f"{p['sens']:.1f}%", ha='center', fontsize=10, fontweight='bold')
bars[-1].set_edgecolor('gold'); bars[-1].set_linewidth(2.5)

# ── Specificity ──
ax = axes[2]
bars = ax.bar(names, [p['spec'] for p in phases], color=colors, edgecolor='black', alpha=0.85)
ax.set_ylim(60, 102)
ax.set_ylabel('Specificity (%)', fontsize=12); ax.set_title('Specificity (Youden-J)', fontsize=13)
ax.grid(axis='y', alpha=0.3)
for i,p in enumerate(phases):
    ax.text(i, p['spec']+0.5, f"{p['spec']:.1f}%", ha='center', fontsize=10, fontweight='bold')
bars[-1].set_edgecolor('gold'); bars[-1].set_linewidth(2.5)

plt.suptitle('PneumoFusionNet — All Phases Comparison\n(Phase 3c: WBC-only ablation highlighted ✨)',
             fontsize=14, y=1.03)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'p3c_full_pipeline_comparison.png'), bbox_inches='tight', dpi=150)
plt.show()
print('✅ Saved all-phases comparison chart.')

In [ ]:
# --- CELL 17: Save Results JSON ---

results = {
    'phase': '3c',
    'description': 'WBC-Only Triple Fusion: Image + Text + WBC Clinical Feature',
    'architecture': {
        'image_encoder':  'DenseNet121+CBAM (frozen from Phase 1.1v4 Fold 2)',
        'text_encoder':   'Bio_ClinicalBERT (last 2 layers unfrozen)',
        'meta_encoder':   f'MLP({N_CLINICAL_FEATURES}->{META_HIDDEN}->{META_HIDDEN}->{META_OUT_DIM})',
        'fusion':         'CrossAttention(img queries text) + concat WBC meta + MLP classifier',
        'fused_dim':      FUSED_DIM,
        'clinical_feats': CLINICAL_FEATURES,
        'n_clinical_features': N_CLINICAL_FEATURES,
        'p2v2_weights_loaded': os.path.exists(P2V2_CKPT)
    },
    'training': {
        'n_train': len(train_df), 'n_val': len(val_df), 'n_test': len(test_df),
        'batch_size': BATCH_SIZE, 'epochs_run': len(history['val_auc']),
        'best_val_auc': round(best_auc, 4),
        'lr_fusion': LR_FUSION, 'lr_bert': LR_BERT, 'lr_meta': LR_META,
        'focal_gamma': FOCAL_GAMMA, 'mixup_alpha': MIXUP_ALPHA
    },
    'results': {
        'test_auc':              round(test_auc, 4),
        'default_acc':           round(ac_d, 4),
        'default_sensitivity':   round(s_d,  4),
        'default_specificity':   round(sp_d, 4),
        'youden_threshold':      round(thresh_youden, 4),
        'youden_acc':            round(ac_y, 4),
        'youden_sensitivity':    round(s_y,  4),
        'youden_specificity':    round(sp_y, 4),
        'clinical_threshold':    round(thresh_clinical, 4),
        'clinical_acc':          round(ac_c, 4),
        'clinical_sensitivity':  round(s_c,  4),
        'clinical_specificity':  round(sp_c, 4),
        'n_test': len(test_labels)
    },
    'comparison': {
        'phase_2v2': {'auc': 0.9490, 'sens': 0.9137, 'spec': 0.8625},
        'phase_3_16feat': {'auc': 0.9841, 'sens': 0.9494, 'spec': 0.9455},
        'phase_3c_wbc_only': {
            'auc':  round(test_auc, 4),
            'sens': round(s_y, 4),
            'spec': round(sp_y, 4)
        },
        'auc_delta_vs_p2v2':       round(test_auc - 0.9490, 4),
        'auc_delta_vs_p3_16feat':  round(test_auc - 0.9841, 4),
        'sens_delta_vs_p2v2':      round(s_y - 0.9137, 4),
        'sens_delta_vs_p3_16feat': round(s_y - 0.9494, 4),
    }
}

out_path = os.path.join(SAVE_DIR, 'phase3c_results.json')
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)

print('='*60)
print('  PHASE 3c (WBC-ONLY) — FINAL RESULTS SUMMARY')
print('='*60)
print(f'Clinical features : {CLINICAL_FEATURES} (n={N_CLINICAL_FEATURES})')
print(f'Test AUC          : {test_auc:.4f}')
print(f'Sensitivity (Y-J) : {s_y*100:.1f}%')
print(f'Specificity (Y-J) : {sp_y*100:.1f}%')
print(f'Accuracy    (Y-J) : {ac_y*100:.1f}%')
print(f'')
print(f'vs Phase 2v2      (Image+Text+0 clinical):')
print(f'  AUC delta  : {test_auc-0.9490:+.4f}')
print(f'  Sens delta : {s_y-0.9137:+.4f}')
print(f'')
print(f'vs Phase 3        (Image+Text+16 clinical):')
print(f'  AUC delta  : {test_auc-0.9841:+.4f}')
print(f'  Sens delta : {s_y-0.9494:+.4f}')
print(f'  (shows how much signal is lost vs full feature set)')
print(f'')
print(f'Results saved to: {out_path}')